# Phase 3 — Gaze Feature EDA

Sanity-check the utterance-level gaze features produced by `features/gaze/build_features.py`.

Checks:
1. Load train / dev / test feature dicts
2. Zero-vector rate
3. Per-feature histogram
4. Correlation matrix
5. Per-emotion mean ± std

In [ ]:
import pickle
import csv
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path("..")
FEAT_NAMES = ['p_face', 'p_out', 'switch_rate', 'entropy', 'target_count', 'p_center']
EMOTION_LABELS = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']

def load_features(split):
    p = ROOT / 'features' / 'gaze' / f'{split}.pkl'
    with p.open('rb') as f:
        return pickle.load(f)

def load_labels(split):
    csv_path = ROOT / 'data' / 'MELD.Raw' / f'{split}_sent_emo.csv'
    if not csv_path.exists():
        csv_path = ROOT / 'data' / 'MELD.Raw' / f'{split}_meld_emo.csv'
    labels = {}
    with csv_path.open() as f:
        reader = csv.DictReader(f)
        for row in reader:
            labels[(int(row['Dialogue_ID']), int(row['Utterance_ID']))] = row['Emotion'].lower()
    return labels

train_feat   = load_features('train')
train_labels = load_labels('train')
print(f'Train features: {len(train_feat)} utterances')

In [ ]:
# Zero-vector rate
zero_count = sum(1 for v in train_feat.values() if v.sum() == 0)
print(f'Zero-vector rate: {100*zero_count/len(train_feat):.2f}% ({zero_count}/{len(train_feat)})')

In [ ]:
# Build DataFrame
rows = []
for (d, u), vec in train_feat.items():
    emo = train_labels.get((d, u), 'unknown')
    rows.append({'emotion': emo, **{FEAT_NAMES[i]: float(vec[i]) for i in range(6)}})
df = pd.DataFrame(rows)
df = df[df['emotion'].isin(EMOTION_LABELS)]
print(df.describe())

In [ ]:
# Histogram of each feature
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for i, feat in enumerate(FEAT_NAMES):
    ax = axes[i // 3][i % 3]
    ax.hist(df[feat].dropna(), bins=40, color='steelblue', alpha=0.8)
    ax.set_title(feat)
    ax.set_xlabel('value')
plt.suptitle('Gaze Feature Distributions (train)')
plt.tight_layout()
plt.savefig(str(ROOT / 'results' / 'figures' / 'eda_gaze_histograms.png'), dpi=120)
plt.show()

In [ ]:
# Correlation matrix
corr = df[FEAT_NAMES].corr()
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Gaze Feature Correlation')
plt.tight_layout()
plt.savefig(str(ROOT / 'results' / 'figures' / 'eda_gaze_corr.png'), dpi=120)
plt.show()

In [ ]:
# Per-emotion mean ± std
stats = df.groupby('emotion')[FEAT_NAMES].agg(['mean', 'std'])
display(stats.round(3))